In [1]:
import os
os.environ['PYSPARK_PYTHON']='python'
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('EcommerceSalesAnalysis').getOrCreate()
print('EcommerceSalesAnalysis-Pyspark start successful!!!!!!!')
# 1. Create PySpark/scala DataFrames for both the sales and products data.
#sales_data
sales_data=[
    (1, 101,'2025-01-15',2),
    (2, 102,'2025-01-17',1),
    (3, 101,'2025-02-10',1),
    (4, 103,'2025-02-20',3),
    (5, 104,'2025-03-05',1),
    (6, 101,'2025-03-12',3),
    (7, 102,'2025-04-01',2),
    (8, 105,'2024-12-20',5),
    (9, 103,'2025-05-21',2),
    (10, 104,'2025-05-30',4)
]
#products_data
products_data=[
    ( 101,'Laptop','Electronics', 1200),
    ( 102, 'Mouse','Electronics', 25),
    ( 103, 'T-Shirt','Apparel', 20),
    ( 104, 'Jeans','Apparel', 75),
    ( 105, 'Book','Books', 15)
]
#sales_data_columns
sales_data_columns=['sale_id','product_id','sale_date','quantity']
# products_data_columns
products_data_columns=['product_id','product_name','category','price']
df_sales=spark.createDataFrame(sales_data,sales_data_columns)
df_products=spark.createDataFrame(products_data,products_data_columns)
print('sales_data:')
df_sales.show()
print('products_data:')
df_products.show()



EcommerceSalesAnalysis-Pyspark start successful!!!!!!!
sales_data:
+-------+----------+----------+--------+
|sale_id|product_id| sale_date|quantity|
+-------+----------+----------+--------+
|      1|       101|2025-01-15|       2|
|      2|       102|2025-01-17|       1|
|      3|       101|2025-02-10|       1|
|      4|       103|2025-02-20|       3|
|      5|       104|2025-03-05|       1|
|      6|       101|2025-03-12|       3|
|      7|       102|2025-04-01|       2|
|      8|       105|2024-12-20|       5|
|      9|       103|2025-05-21|       2|
|     10|       104|2025-05-30|       4|
+-------+----------+----------+--------+

products_data:
+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics| 1200|
|       102|       Mouse|Electronics|   25|
|       103|     T-Shirt|    Apparel|   20|
|       104|       Jeans|    Apparel|   75|
|       105|        Book|      Book

In [3]:
print(df_sales.schema["sale_id"].dataType)
print(df_sales.schema["product_id"].dataType)
print(df_sales.schema["sale_date"].dataType)
print(df_sales.schema["quantity"].dataType)
print(df_sales.printSchema())
print(df_products.printSchema())



LongType()
LongType()
StringType()
LongType()
root
 |-- sale_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- quantity: long (nullable = true)

None
root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: long (nullable = true)

None


In [3]:
df_sales.schema["sale_date"].dataType


StringType()

In [4]:
# 2:Filter the sales data to include only records from the year 2025.
# df_sales.filter(df_sales.sale_date=='2025-01-15')
df_sales_date_2025=df_sales.filter(df_sales.sale_date.like('2025%'))
df_sales_date_2025.show()


+-------+----------+----------+--------+
|sale_id|product_id| sale_date|quantity|
+-------+----------+----------+--------+
|      1|       101|2025-01-15|       2|
|      2|       102|2025-01-17|       1|
|      3|       101|2025-02-10|       1|
|      4|       103|2025-02-20|       3|
|      5|       104|2025-03-05|       1|
|      6|       101|2025-03-12|       3|
|      7|       102|2025-04-01|       2|
|      9|       103|2025-05-21|       2|
|     10|       104|2025-05-30|       4|
+-------+----------+----------+--------+



In [5]:
# 3. Join the filtered sales data with the products data on product_id
df_join_2025_products=df_sales_date_2025.join(df_products,on='product_id',how='inner')
print('df_join_2025_products:')
df_join_2025_products.show()

df_join_2025_products:
+----------+-------+----------+--------+------------+-----------+-----+
|product_id|sale_id| sale_date|quantity|product_name|   category|price|
+----------+-------+----------+--------+------------+-----------+-----+
|       101|      1|2025-01-15|       2|      Laptop|Electronics| 1200|
|       101|      3|2025-02-10|       1|      Laptop|Electronics| 1200|
|       101|      6|2025-03-12|       3|      Laptop|Electronics| 1200|
|       102|      2|2025-01-17|       1|       Mouse|Electronics|   25|
|       102|      7|2025-04-01|       2|       Mouse|Electronics|   25|
|       103|      4|2025-02-20|       3|     T-Shirt|    Apparel|   20|
|       103|      9|2025-05-21|       2|     T-Shirt|    Apparel|   20|
|       104|      5|2025-03-05|       1|       Jeans|    Apparel|   75|
|       104|     10|2025-05-30|       4|       Jeans|    Apparel|   75|
+----------+-------+----------+--------+------------+-----------+-----+



In [6]:
# 4. Calculate the total revenue for each transaction (quantity * price)
from pyspark.sql.functions import expr
print('total revenue for each transaction:')

df_each_revenue=df_join_2025_products.withColumn(
    'df_each_revenue',
    expr('quantity*price')
)
df_each_revenue.show()

total revenue for each transaction:
+----------+-------+----------+--------+------------+-----------+-----+---------------+
|product_id|sale_id| sale_date|quantity|product_name|   category|price|df_each_revenue|
+----------+-------+----------+--------+------------+-----------+-----+---------------+
|       101|      1|2025-01-15|       2|      Laptop|Electronics| 1200|           2400|
|       101|      3|2025-02-10|       1|      Laptop|Electronics| 1200|           1200|
|       101|      6|2025-03-12|       3|      Laptop|Electronics| 1200|           3600|
|       102|      2|2025-01-17|       1|       Mouse|Electronics|   25|             25|
|       102|      7|2025-04-01|       2|       Mouse|Electronics|   25|             50|
|       103|      4|2025-02-20|       3|     T-Shirt|    Apparel|   20|             60|
|       103|      9|2025-05-21|       2|     T-Shirt|    Apparel|   20|             40|
|       104|      5|2025-03-05|       1|       Jeans|    Apparel|   75|             

In [7]:
from pyspark.sql.functions import sum,row_number,rank,desc,col
df_total_revenue=df_each_revenue.agg(
    sum('df_each_revenue')
)
print('total revenue is :')
df_total_revenue.show()


total revenue is :
+--------------------+
|sum(df_each_revenue)|
+--------------------+
|                7750|
+--------------------+



In [8]:
# 5. Group the data by category and product_name and calculate the total revenue for each product.

df_each_revenue.show()

+----------+-------+----------+--------+------------+-----------+-----+---------------+
|product_id|sale_id| sale_date|quantity|product_name|   category|price|df_each_revenue|
+----------+-------+----------+--------+------------+-----------+-----+---------------+
|       101|      1|2025-01-15|       2|      Laptop|Electronics| 1200|           2400|
|       101|      3|2025-02-10|       1|      Laptop|Electronics| 1200|           1200|
|       101|      6|2025-03-12|       3|      Laptop|Electronics| 1200|           3600|
|       102|      2|2025-01-17|       1|       Mouse|Electronics|   25|             25|
|       102|      7|2025-04-01|       2|       Mouse|Electronics|   25|             50|
|       103|      4|2025-02-20|       3|     T-Shirt|    Apparel|   20|             60|
|       103|      9|2025-05-21|       2|     T-Shirt|    Apparel|   20|             40|
|       104|      5|2025-03-05|       1|       Jeans|    Apparel|   75|             75|
|       104|     10|2025-05-30| 

In [9]:
# 5. Group the data by category and product_name and calculate the total revenue for each product.

each_total_revenue=df_each_revenue.groupBy('category','product_name').agg(
    sum('df_each_revenue').alias('each_total_revenue')
)
each_total_revenue.show()

+-----------+------------+------------------+
|   category|product_name|each_total_revenue|
+-----------+------------+------------------+
|    Apparel|     T-Shirt|               100|
|    Apparel|       Jeans|               375|
|Electronics|       Mouse|                75|
|Electronics|      Laptop|              7200|
+-----------+------------+------------------+



In [10]:
# 6. For each category, find the top 2 products with the highest total revenue.

from pyspark.sql.window import Window
window_spec = Window.partitionBy("category").orderBy(desc("each_total_revenue"))
ranked_df = each_total_revenue.withColumn(
    "rn",  
    row_number().over(window_spec)
).filter(col("rn") <= 2)
ranked_df.show()


+-----------+------------+------------------+---+
|   category|product_name|each_total_revenue| rn|
+-----------+------------+------------------+---+
|    Apparel|       Jeans|               375|  1|
|    Apparel|     T-Shirt|               100|  2|
|Electronics|      Laptop|              7200|  1|
|Electronics|       Mouse|                75|  2|
+-----------+------------+------------------+---+



In [11]:
#close
spark.stop()